In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Smart MCQ Solver Challenge — End-to-End Solution
### MAP@3 ranking of top-3 answers (A–E) for knowledge-based MCQs

**Pipeline overview**

| # | Model | Category | Idea |
|---|-------|----------|------|
| 1 | TF-IDF + PyTorch MLP | **Built from scratch** | No pretrained weights anywhere; learns purely from the ~2,000 training rows |
| 2 | DeBERTa-v3 (`AutoModelForMultipleChoice`) | **Pretrained, fine-tuned** | Transfer learning — pretrained language + world knowledge, adapted to this task |
| 3 | XGBoost on engineered similarity features (TF-IDF sim, Sentence-Transformer sim, lexical overlap, etc.) | **Additional model of choice** | Tree-based, feature-driven — a structurally different failure mode from 1 & 2 |
| — | Weighted ensemble of the three | **Final submission** | Combines probability outputs, tuned on a held-out validation split |

**Notebook structure**
1. Setup & data loading
2. EDA (light)
3. MAP@3 metric implementation
4. Train/validation split
5. Model 1 — from-scratch TF-IDF + MLP
6. Model 2 — pretrained DeBERTa-v3 fine-tuned as multiple-choice classifier
7. Model 3 — XGBoost on engineered similarity features
8. Ensembling + local MAP@3 evaluation
9. Final inference on `test.csv` + submission file
10. (Optional, commented out) Zero-shot LLM prompting extension

> Upload `train.csv`, `test.csv`, and `sample_submission.csv` to the Colab working directory (or mount Google Drive) before running.


## 1. Setup

In [2]:
# Run this once per session.
# NOTE: Kaggle's base image already ships compatible, modern versions of numpy, pandas,
# scikit-learn, and torch (with correct GPU/CUDA support) -- do NOT pin/reinstall those,
# it breaks scipy/sklearn (numpy 1.26 vs scipy built for numpy 2.x -> "numpy.strings" error).
# We only install what is actually missing from the Kaggle image.

!pip install --upgrade pip -q

!pip install \
    transformers==4.46.0 \
    datasets==3.1.0 \
    accelerate==1.1.0 \
    sentence-transformers==3.0.1 \
    xgboost==2.1.1 \
    wandb \
    --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 6.5 MB/s eta 0:00:0000:0100:01
Reason for being yanked: This version unfortunately does not work with 3.8 but we did not drop the support yet
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
tpot 1.1.0 requires dill>=0.3.9, but you have dill 0.3.8 which is incompatible.
tpot 1.1.0 requires xgboost>=3.0.0, but you have xgboost 2.1.1 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you have numba-cuda 0.30.2 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.


# Environment Verification Script

## Purpose
Verifies all ML/DL library installations and GPU availability in Kaggle/Colab environment.


In [3]:
# Verify installations
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
import xgboost
import sentence_transformers
import datasets
import accelerate

print("✅ All imports successful!")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"XGBoost: {xgboost.__version__}")
print(f"Sentence-Transformers: {sentence_transformers.__version__}")
print(f"Datasets: {datasets.__version__}")
print(f"Accelerate: {accelerate.__version__}")

# Check GPU availability
print(f"\nGPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

2026-08-07 08:04:00.808508: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1786089840.978860     127 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1786089841.030387     127 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1786089841.423786     127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786089841.423823     127 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1786089841.423825     127 computation_placer.cc:177] computation placer alr

✅ All imports successful!
NumPy: 2.4.6
Pandas: 2.3.3
Scikit-learn: 1.6.1
PyTorch: 2.10.0+cu128
Transformers: 4.46.0
XGBoost: 2.1.1
Sentence-Transformers: 3.0.1
Datasets: 3.1.0
Accelerate: 1.1.0

GPU Available: True
GPU Name: Tesla T4



# Initializes Weights & Biases (wandb) logging for the MCQ Solver project.
- Fetches the API key securely from Kaggle Secrets.
- Sets project names and groups for better experiment tracking.
- Fixes the `SyntaxWarning: invalid escape sequence` by using raw strings.
- Automatically creates the `.netrc` file for wandb authentication.


In [4]:

import wandb
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

WANDB_PROJECT = "smart-mcq-solver"   # all 3 model runs + ensemble will appear under this one project
WANDB_GROUP   = "mcq-ensemble-v1"    # groups them together so they're easy to compare on wandb.ai


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f3004169 (23f3004169-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Sets up the core environment and deterministic configurations for the training pipeline.
- Imported all necessary libraries (PyTorch, Sklearn, Pandas, Numpy).
- Standardized global random seeds (42) for reproducibility.
- Configured automatic device detection (GPU/CPU) for PyTorch.
- Suppressed minor warnings to keep logs clean.

In [1]:
import os, re, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

Using device: cuda


In [4]:
import os
import pandas as pd

# Detect environment automatically to handle path changes
def get_data_path():
    """
    Checks multiple common directories to find the data.
    Returns the base directory path if found, otherwise raises a clear error.
    """
    # Option 1: Kaggle
    kaggle_path = "/kaggle/input/competitions/smart-mcq-solver-challenge/"
    if os.path.exists(kaggle_path):
        print(f"📂 Environment detected: Kaggle (Using {kaggle_path})")
        return kaggle_path
    
DATA_DIR = get_data_path()

# ==========================================
# 2. Safe File Loading with Existence Checks
# ==========================================
def load_csv_safe(filename):
    """Loads a CSV file and raises a specific error if it's missing."""
    file_path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"❌ Missing file: '{file_path}'. Check your DATA_DIR.")
    return pd.read_csv(file_path)

# Load the files
train = load_csv_safe("train.csv")
test = load_csv_safe("test.csv")
sample_submission = load_csv_safe("sample_submission.csv")

📂 Environment detected: Kaggle (Using /kaggle/input/competitions/smart-mcq-solver-challenge/)


In [5]:
# 3. Data Validation & Options Setup
# ==========================================
OPTIONS = ["A", "B", "C", "D", "E"]

print("=" * 50)
print("✅ Data loaded successfully!")
print(f"Train Shape: {train.shape}")
print(f"Test Shape:  {test.shape}")
print(f"Submission Shape: {sample_submission.shape}")
print("=" * 50)

# Quick check to ensure the answer column exists (if it does) and contains valid options
if 'answer' in train.columns:
    unique_answers = train['answer'].dropna().unique()
    missing_labels = [ans for ans in unique_answers if ans not in OPTIONS]
    if missing_labels:
        print(f"⚠️ Warning: The 'answer' column contains values outside A-E: {missing_labels}")


✅ Data loaded successfully!
Train Shape: (2000, 8)
Test Shape:  (500, 7)
Submission Shape: (500, 2)


In [6]:
# 4. Data Preview
# ==========================================
print("\n📋 Preview of Training Data (First 5 rows):")
display(train.head()) # Works natively in Jupyter/Colab/Kaggle (use print(train.head()) in pure scripts)

print("\n📋 Preview of Test Data (First 5 rows):")
display(test.head())


📋 Preview of Training Data (First 5 rows):


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A



📋 Preview of Test Data (First 5 rows):


,id,prompt,A,B,C,D,E
0,1,Pick the best possible answer: What is the rel...,"For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p...","For every eigenstate of one Hamiltonian, its p..."
1,2,"What is the estimated redshift of CEERS-93316,...","Approximately z = 6.0, corresponding to 1 bill...","Approximately z = 16.7, corresponding to 235.8...","Approximately z = 3.0, corresponding to 5 bill...","Approximately z = 10.0, corresponding to 13 bi...","Approximately z = 13.0, corresponding to 30 bi..."
2,3,Pick the best possible answer: What is the rea...,The sun appears yellowish due to a reflection ...,"The longer wavelengths of light, such as red a...",The sun appears yellowish due to the scatterin...,The sun emits a yellow light due to its own sp...,The atmosphere absorbs the shorter wavelengths...
3,4,What is the significance of the redshift-dista...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...,Observations of the redshift-distance relation...
4,5,What is the Landau-Lifshitz-Gilbert equation u...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...,The Landau-Lifshitz-Gilbert equation is a diff...
